# Python BITCOIN TAPROOT

In [ ]:
# Librerias

from bitcoinutils.setup import setup
from bitcoinutils.keys import PrivateKey, P2trAddress
from bitcoinutils.transactions import Transaction, TxInput, TxOutput, TxWitnessInput
from bitcoinutils.script import Script
from bitcoinutils.keys import P2wpkhAddress

import hashlib
setup("regtest")

'regtest'

In [3]:
def sha256_to_privkey_mod(text: str) -> int:
    N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
    h = hashlib.sha256(text.encode()).digest()
    k = int.from_bytes(h, "big")
    return (k % (N - 1)) + 1

def sk_to_wif(sk_int: int, compressed: bool = True) -> str:
    priv_bytes = sk_int.to_bytes(32, 'big')
    prefix = b'\xef'
    extended_key = prefix + priv_bytes
    if compressed:
        extended_key += b'\x01'
    first_sha = hashlib.sha256(extended_key).digest()
    second_sha = hashlib.sha256(first_sha).digest()
    checksum = second_sha[:4]
    final_key = extended_key + checksum
    return b58encode(final_key)

def b58encode(v: bytes) -> str:
    alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
    n = int.from_bytes(v, "big")
    res = ""
    while n > 0:
        n, r = divmod(n, 58)
        res = alphabet[r] + res
    pad = 0
    for b in v:
        if b == 0: pad += 1
        else: break
    return (alphabet[0] * pad) + res

seed_sk = sha256_to_privkey_mod("hola mundo")
wif_key = sk_to_wif(seed_sk)

print(f"Private Key (Hex): {hex(seed_sk)[2:]}")
print(f"WIF (Regtest):     {wif_key}")

Private Key (Hex): b894166d3336435c800bea36ff21b29eaa801a52f584c006c49289a0dcf6e30
WIF (Regtest):     cMy8HX59BW3CRJL9Sc284dsFkhGGR7AQFPWZMJSnproo6YJGsQnE


In [4]:
sk = PrivateKey.from_wif(wif_key)
pub = sk.get_public_key()
addr = pub.get_taproot_address()
print(addr.to_string())

bcrt1puj7qwquetfzgp670q50dh7jy2550fn5efhf0wksnyshhttzqrhjqm0y9pt


## PAY FROM THE ADDRESS

Se enviaron 25 BTC a esta address desde el nodo bitcoin-core.
Ahora se pagaran enviaran 10 a la misma dirección pagando un fee

In [5]:
txid_in = '5b2678dd20cfb3f50279e9608dd20ce7f1f46912c0fa7f0afd9e33e6224a44b8' 
amount_in = 25*100_000_000
amount_out = 10*100_000_000
fee = 1000
change = amount_in - amount_out - fee

addr_out = "bcrt1qee3gcq3kv56x30qpx8x2eytwflh8wq0xx3zsyn"
addr_out_obj = P2wpkhAddress(addr_out)
addr_change = addr

# CONSTRUIMOS

txin = TxInput(txid_in, 1)
out1 = TxOutput(amount_out, addr_out_obj.to_script_pub_key())
out2 = TxOutput(change, addr_change.to_script_pub_key())

tx = Transaction(
    [txin],
    [out1, out2],
    has_segwit=True
)

utxo_scripts = [addr.to_script_pub_key()]
amounts = [amount_in]

sig = sk.sign_taproot_input(tx, 0, utxo_scripts, amounts)
tx.witnesses.append(TxWitnessInput([sig]))

print(tx.serialize())


02000000000101b8444a22e6339efd0a7ffac01269f4f1e70cd28d60e97902f5b3cf20dd78265b0100000000fdffffff0200ca9a3b00000000160014ce628c0236653468bc0131ccac916e4fee7701e6182b685900000000225120e4bc0703995a4480ebcf051edbfa445528f4ce994dd2f75a13242f75ac401de40140bc54a38d45e92ef51de4f1e92c121278ce91e403e3c076c695e914192a95e4230a715ef6d9fe7c6aea30dcd8d1cdce9bcf9236794c37a7a3dadd5be5c3e77bbc00000000


# Taproot con OP RETURN

Ahora que ha funcionado bien sin problema, probamos una construcción usando un op return para firmar.

In [6]:
txid_in2 = '769574085291b5870a85cd682883af89a8e6cae66302701c304a756d5350c792' 
amount_in = 15*100_000_000 - 1000
amount_out = 10*100_000_000
fee = 1000
change = amount_in - amount_out - fee

addr_out = "bcrt1puj7qwquetfzgp670q50dh7jy2550fn5efhf0wksnyshhttzqrhjqm0y9pt"
addr_change = addr

txin = TxInput(txid_in2, 1)
out1 = TxOutput(amount_out, addr.to_script_pub_key())

data = "hola mundo cien año de soledad".encode().hex()
out2 = TxOutput(0, Script(["OP_RETURN", data]))

out3 = TxOutput(change, addr.to_script_pub_key())

tx = Transaction(
    [txin],
    [out1, out2, out3],
    has_segwit=True
)

utxo_scripts = [addr.to_script_pub_key()]
amounts = [amount_in]

sig = sk.sign_taproot_input(tx, 0, utxo_scripts, amounts)
tx.witnesses.append(TxWitnessInput([sig]))
print(tx.serialize())


0200000000010192c750536d754a301c700263e6cae6a889af832868cd850a87b59152087495760100000000fdffffff0300ca9a3b00000000225120e4bc0703995a4480ebcf051edbfa445528f4ce994dd2f75a13242f75ac401de40000000000000000216a1f686f6c61206d756e646f206369656e2061c3b16f20646520736f6c65646164305dcd1d00000000225120e4bc0703995a4480ebcf051edbfa445528f4ce994dd2f75a13242f75ac401de4014093cab13a648c25a70c3705342f5c989461e0dfc5b909cf1927e85806edfba1166e502433b3e666918d6c05313200b88590c5a1744d2ce0665a3f0980141667e600000000


In [7]:
print(bytes.fromhex("686f6c61206d756e646f").decode('utf-8'))

hola mundo


## Pago a una Dirección Taproot con ScriptPath


generamos una address que contenga en su tweak un scriptpath.

Este es el único script:

OP_SHA256 <hash_del_secreto> OP_EQUALVERIFY <pubkey_xonly_de_bob> OP_CHECKSIG



In [8]:
seed_bob = sha256_to_privkey_mod("hola mundo bob")
wif_key_bob = sk_to_wif(seed_bob)
bob_sk = PrivateKey.from_wif(wif_key_bob)

bob_pub = bob_sk.get_public_key()

print("Bob WIF:", bob_sk.to_wif())
print("Bob xonly:", bob_pub.to_x_only_hex())

secret = b"mi secreto 123"
secret_hash = hashlib.sha256(secret).hexdigest()

print("secret:", secret)
print("sha256(secret):", secret_hash)



Bob WIF: cSRNz6rovaZXp2UzWSzos2M7v6cpgaRWVyXanSaeoZm6pgyqFmpV
Bob xonly: 2611b842ea1879624eae5db8307a495804d1da2275c35205be6a951a26004acd
secret: b'mi secreto 123'
sha256(secret): 1a92f51640fbb587cf184d04e980bf580296155df5f54f88658b1f184cb3e069


In [9]:
hashlock_leaf = Script([
    "OP_SHA256",
    secret_hash,
    "OP_EQUALVERIFY",
    bob_pub.to_x_only_hex(),
    "OP_CHECKSIG"
])

print(hashlock_leaf)

taproot_addr = bob_pub.get_taproot_address([hashlock_leaf])
print(taproot_addr.to_string())

['OP_SHA256', '1a92f51640fbb587cf184d04e980bf580296155df5f54f88658b1f184cb3e069', 'OP_EQUALVERIFY', '2611b842ea1879624eae5db8307a495804d1da2275c35205be6a951a26004acd', 'OP_CHECKSIG']
bcrt1pdvyhevpalfq4gdhnt4ategqnhnfrs8rqw0cqwa6f6nny5qur6zmqj67wn3


In [10]:
print(hashlock_leaf)
print(hashlock_leaf.to_hex())
print(taproot_addr.to_string())

['OP_SHA256', '1a92f51640fbb587cf184d04e980bf580296155df5f54f88658b1f184cb3e069', 'OP_EQUALVERIFY', '2611b842ea1879624eae5db8307a495804d1da2275c35205be6a951a26004acd', 'OP_CHECKSIG']
a8201a92f51640fbb587cf184d04e980bf580296155df5f54f88658b1f184cb3e06988202611b842ea1879624eae5db8307a495804d1da2275c35205be6a951a26004acdac
bcrt1pdvyhevpalfq4gdhnt4ategqnhnfrs8rqw0cqwa6f6nny5qur6zmqj67wn3


Ahora que ya tiene fondos de 25 BTC vamos a pagar 10 BTC a una address del nodo.


In [11]:
txid_in = "28fe72a9bdbc0b45d0d7e84e618797290d134045acba0233a5a30f442e32d52c"
amount_in = 25 * 100_000_000
bob_addr = bob_pub.get_taproot_address([hashlock_leaf])


addr_out = "bcrt1qee3gcq3kv56x30qpx8x2eytwflh8wq0xx3zsyn"  # dirección del nodo
amount_out = 10 * 100_000_000
fee = 1000
change = amount_in - amount_out - fee
addr_out_obj = P2wpkhAddress(addr_out)

txin = TxInput(txid_in, 0)
out1 = TxOutput(amount_out, addr_out_obj.to_script_pub_key())
out2 = TxOutput(change, bob_addr.to_script_pub_key())

tx = Transaction(
    [txin],
    [out1, out2],
    has_segwit=True
)

utxo_scripts = [bob_addr.to_script_pub_key()]
amounts = [amount_in]

sig = bob_sk.sign_taproot_input(
    tx,
    0,
    utxo_scripts,
    amounts,
    script_path=False,
    tapleaf_scripts=[hashlock_leaf],
)

tx.witnesses.append(TxWitnessInput([sig]))

print(tx.serialize())
print("TXID:", tx.get_txid())

020000000001012cd5322e440fa3a53302baac4540130d299787614ee8d7d0450bbcbda972fe280000000000fdffffff0200ca9a3b00000000160014ce628c0236653468bc0131ccac916e4fee7701e6182b6859000000002251206b097cb03dfa415436f35d7abca013bcd2381c6073f0077749d4e64a0383d0b6014074c06b623db7e908c50f92787f30b774a9c5269549cdde05b5d2e0294352a38f8a5be20260fc7f3467f4542f33046794e84362ee316cf0ab69e7183aa6b1554300000000
TXID: ff2fc9525de184f312d148ba3baddc9d0264d31f3d3c2de96d9b1cb94cfb2b65


## Gastar un Script Taproot con un OP RETURN

In [12]:

txid_in = "ff2fc9525de184f312d148ba3baddc9d0264d31f3d3c2de96d9b1cb94cfb2b65"
amount_in = (15 * 100_000_000) - 1000

bob_addr = bob_pub.get_taproot_address([hashlock_leaf])

addr_out = "bcrt1qee3gcq3kv56x30qpx8x2eytwflh8wq0xx3zsyn"
addr_out_obj = P2wpkhAddress(addr_out)

amount_out = 5 * 100_000_000 
fee = 1000

# dato a grabar
opret_data = b"JuanpyBTC hola mundo"
opret_hex = opret_data.hex()

# salida OP_RETURN
opret_script = Script(["OP_RETURN", opret_hex])
opret_out = TxOutput(0, opret_script)

change = amount_in - amount_out - fee

txin = TxInput(txid_in, 1)
out1 = TxOutput(amount_out, addr_out_obj.to_script_pub_key())
out2 = TxOutput(change, bob_addr.to_script_pub_key())

tx = Transaction(
    [txin],
    [out1, opret_out, out2],
    has_segwit=True
)

utxo_scripts = [bob_addr.to_script_pub_key()]
amounts = [amount_in]

sig = bob_sk.sign_taproot_input(
    tx,
    0,
    utxo_scripts,
    amounts,
    script_path=False,
    tapleaf_scripts=[hashlock_leaf],
)

tx.witnesses.append(TxWitnessInput([sig]))

print("OP_RETURN script:", opret_script.to_hex())
print(tx.serialize())
print("TXID:", tx.get_txid())

OP_RETURN script: 6a144a75616e707942544320686f6c61206d756e646f
02000000000101652bfb4cb91c9b6de92d3c3d1fd364029ddcad3bba48d112f384e15d52c92fff0100000000fdffffff030065cd1d00000000160014ce628c0236653468bc0131ccac916e4fee7701e60000000000000000166a144a75616e707942544320686f6c61206d756e646f30c29a3b000000002251206b097cb03dfa415436f35d7abca013bcd2381c6073f0077749d4e64a0383d0b6014060d73b7078276d2f553c60a71f33b4117cbb1bb17c88c0b052a47987b7d5837f39bc4e28e8c7b5996690b67d4951add5e8583dd4706f1d7447259d9582c2413300000000
TXID: caa1f33b753837d883728c148789f09d66ce22b78af353070a373b9feb76bf32


In [1]:
print(bytes.fromhex("4a75616e7079425443").decode('utf-8'))

JuanpyBTC
